In [10]:
import pandas as pd
import altair as alt
df = pd.read_csv('website/cleaned_listings.csv')
df.columns

Index(['price', 'number_of_reviews', 'latitude', 'longitude', 'room_type',
       'minimum_nights', 'availability_365', 'review_scores_rating', 'license',
       'neighbourhood_cleansed', 'calculated_host_listings_count', 'bedrooms',
       'name', 'number_of_reviews_ltm', 'host_is_superhost',
       'estimated_occupancy_l365d', 'estimated_revenue_l365d', 'has_license'],
      dtype='object')

In [11]:
# filter out outliers
df = df[df['price'] < 1000]

In [12]:
df.neighbourhood_cleansed.unique()

array(['East Boston', 'Roxbury', 'Beacon Hill', 'Back Bay', 'North End',
       'Dorchester', 'Charlestown', 'Jamaica Plain', 'Downtown',
       'South Boston', 'West Roxbury', 'Roslindale', 'South End',
       'Mission Hill', 'Brighton', 'Fenway', 'Allston', 'Hyde Park',
       'West End', 'Mattapan', 'Chinatown', 'Bay Village',
       'South Boston Waterfront', 'Leather District',
       'Longwood Medical Area'], dtype=object)

In [13]:
df = df[df['neighbourhood_cleansed'] != 'Leather District']

In [14]:
# Create aggregated data
neighborhood_stats = df.groupby('neighbourhood_cleansed').agg({
    'price': 'median',
    'estimated_revenue_l365d': 'median',
    'estimated_occupancy_l365d': 'mean',
}).reset_index()


scatterplot1 = alt.Chart(neighborhood_stats).mark_circle(size=200, opacity=0.8).encode(
    x=alt.X('price:Q', 
            title='Median Price per Night ($)',
            scale=alt.Scale(domain=[0, 500])),
    y=alt.Y('estimated_revenue_l365d:Q', 
            title='Median Estimated Revenue (Last 365 Days)'),
    color=alt.Color('estimated_occupancy_l365d:Q', 
                    title='Average Occupancy (l365d)'),
    tooltip=[
        alt.Tooltip('neighbourhood_cleansed:N', title='Neighborhood'),
        alt.Tooltip('price:Q', title='Median Price', format='$.2f'),
        alt.Tooltip('estimated_revenue_l365d:Q', title='Median Revenue (Last 365 Days)', format='$,.0f'),
        alt.Tooltip('estimated_occupancy_l365d:Q', title='Mean Estimated Occupancy (Last 365 Days)')
    ]
).properties(
    width=700,
    height=400,
    title='Median Price vs Revenue by Neighborhood'
)

In [15]:
all_neighborhoods = list(df['neighbourhood_cleansed'].unique())

In [16]:
input_dropdown = alt.binding_select(options = all_neighborhoods,
                                    name = 'Neighborhood: ')
selection = alt.selection_point(fields=['neighbourhood_cleansed'], bind=input_dropdown, value = 'Back Bay')


scatterplot2 = alt.Chart(df).mark_circle(size=60, opacity=0.6).encode(
    x=alt.X('price:Q', 
            title='Price per Night ($)',
            scale=alt.Scale(domain=[0, 1000])),
    y=alt.Y('estimated_revenue_l365d:Q', 
            title='Estimated Revenue (Last 365 Days)'),
    color=alt.Color('estimated_occupancy_l365d:Q', 
                    title='Estimated Occupancy'),
    opacity=alt.condition(selection, alt.value(1), alt.value(0)),
    tooltip=[
        alt.Tooltip('name:N', title='Name'),
        alt.Tooltip('neighbourhood_cleansed:N', title='Neighborhood'),
        alt.Tooltip('price:Q', title='Price', format='$.2f'),
        alt.Tooltip('estimated_revenue_l365d:Q', title='Revenue (l365d)', format='$,.0f'),
        alt.Tooltip('estimated_occupancy_l365d:Q', title='Estimated Occupancy (l365d)')
    ]
).add_params(selection).transform_filter(
    selection).properties(
    width=700,
    height=400,
    title='Price vs Revenue by Neighborhood'
)

In [17]:
s1_json = scatterplot1.to_json()
s2_json = scatterplot2.to_json()
with open('website/t1-scatterplot1_spec.json', 'w') as f:
    f.write(s1_json)
    
with open('website/t1-scatterplot2_spec.json', 'w') as f:
    f.write(s2_json)